# Job → Skills → Courses Matching Pipeline

**Flow:**
1. Load data (Skills, Courses, Jobs) from Excel
2. Embed everything with sentence-transformers (checkpointed)
3. Match jobs → skills via cosine similarity + threshold
4. Match missing skills → courses via cosine similarity + threshold
5. Export final mapping: Job → Courses

## 0. Install & Imports

In [20]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── CONFIG ──────────────────────────────────────────────────────────────────

EXCEL_PATH = "data/Professional_growth_exercise.xlsx"   # <-- update if needed
CACHE_DIR  = "./embedding_cache"                    # checkpoints go here
os.makedirs(CACHE_DIR, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

JOB_SKILL_THRESHOLD   = 0.35
SKILL_COURSE_THRESHOLD = 0.35

# Safety caps (avoid too many matches per item)
MAX_SKILLS_PER_JOB    = 15
MAX_COURSES_PER_SKILL = 5

print("Config OK")

Config OK


## 1. Load Data

In [21]:

df_skills_enriched = pd.read_csv("data/skills_esco_matched.csv")
df_courses = pd.read_excel(EXCEL_PATH, sheet_name="Courses")  # Corso, SkillUri
df_jobs    = pd.read_excel(EXCEL_PATH, sheet_name="Jobs")     # esco_job_title_eng

# Basic cleanup
df_courses = df_courses.dropna(subset=["Corso"]).drop_duplicates()
df_jobs    = df_jobs.dropna(subset=["esco_job_title_eng"]).drop_duplicates(subset="esco_job_title_eng")


skills_list = df_skills_enriched["skill_text_enriched"].tolist()
skill_uris  = df_skills_enriched["SkillUri"].tolist()
courses_list = df_courses["Corso"].tolist()
jobs_list    = df_jobs["esco_job_title_eng"].tolist()

print(f"Skills:  {len(skills_list)}")
print(f"Courses: {len(courses_list)}")
print(f"Jobs:    {len(jobs_list)}")

df_skills_enriched.head(3)

Skills:  15037
Courses: 10756
Jobs:    2986


,SkillUri,Skill,match_score,matched,esco_uri,esco_label,esco_altLabels,esco_description,skill_text_enriched
0,2.A.1.b,Active Listening,0.7565,True,http://data.europa.eu/esco/skill/a17286c5-238d...,listen actively,maintain active listening\nactive listening\na...,"Give attention to what other people say, patie...","listen actively. maintain active listening, ac..."
1,2.C.3.b,Engineering and Technology,0.5937,True,http://data.europa.eu/esco/skill/14764a75-3104...,telecommunications engineering,telecoms engineering\ntele-communications engi...,Discipline that combines computer science with...,telecommunications engineering. telecoms engin...
2,2.C.4.a,Mathematics,0.6605,True,http://data.europa.eu/esco/skill/4339176e-3acd...,mathematics,numeracy\nquantitative data\nsolid modeling\nm...,Mathematics is the study of topics such as qua...,"mathematics. numeracy, quantitative data, soli..."


## 2. Load Model

In [ ]:
import os
model = SentenceTransformer(MODEL_NAME)
print(f"Loaded: {MODEL_NAME}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7836.51it/s]


Loaded: paraphrase-multilingual-MiniLM-L12-v2


## 3. Embed Everything (with checkpoints)

Each embedding block checks for a cached `.npy` file first.
Delete the cache file to force a re-embed.

In [23]:
def embed_with_cache(texts, cache_path, model, batch_size=64, show_progress=True):
    """Embed a list of texts, loading from cache if it exists."""
    if os.path.exists(cache_path):
        print(f"  ↳ loading from cache: {cache_path}")
        return np.load(cache_path)
    print(f"  ↳ embedding {len(texts)} items...")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=show_progress,
        normalize_embeddings=True   # normalized → cosine sim = dot product (faster)
    )
    np.save(cache_path, embeddings)
    print(f"  ↳ saved to {cache_path}")
    return embeddings

In [24]:
print("Embedding skills...")
emb_skills = embed_with_cache(
    skills_list,
    os.path.join(CACHE_DIR, "emb_skills.npy"),
    model
)

print("Embedding courses...")
emb_courses = embed_with_cache(
    courses_list,
    os.path.join(CACHE_DIR, "emb_courses.npy"),
    model
)

print("Embedding jobs...")
emb_jobs = embed_with_cache(
    jobs_list,
    os.path.join(CACHE_DIR, "emb_jobs.npy"),
    model
)

print(f"\nShapes → skills: {emb_skills.shape}, courses: {emb_courses.shape}, jobs: {emb_jobs.shape}")

Embedding skills...
  ↳ loading from cache: ./embedding_cache/emb_skills.npy
Embedding courses...
  ↳ loading from cache: ./embedding_cache/emb_courses.npy
Embedding jobs...
  ↳ loading from cache: ./embedding_cache/emb_jobs.npy

Shapes → skills: (15037, 384), courses: (10756, 384), jobs: (2986, 384)


## 4. Match Jobs → Skills

For each job, find skills above the similarity threshold (up to MAX_SKILLS_PER_JOB).
These are the **required** skills for that job.

In [25]:
def match_by_threshold(query_embs, candidate_embs, candidate_labels, threshold, max_matches):
    """
    For each query, return candidates above threshold (sorted by score desc).
    Returns a list of lists: [[{label, score}, ...], ...]
    """
    # Since embeddings are normalized, dot product == cosine similarity
    sim_matrix = query_embs @ candidate_embs.T   # shape: (n_queries, n_candidates)
    results = []
    for sims in sim_matrix:
        # Get indices above threshold, sorted by score
        above = np.where(sims >= threshold)[0]
        above_sorted = above[np.argsort(-sims[above])]
        above_sorted = above_sorted[:max_matches]  # cap
        matches = [{"label": candidate_labels[i], "score": float(sims[i])} for i in above_sorted]
        results.append(matches)
    return results

In [26]:
job_to_skills = match_by_threshold(
    emb_jobs, emb_skills, skills_list,
    threshold=JOB_SKILL_THRESHOLD,
    max_matches=MAX_SKILLS_PER_JOB
)

# Quick look
for job, matches in zip(jobs_list[:3], job_to_skills[:3]):
    print(f"\n[{job}]")
    for m in matches[:5]:
        print(f"  {m['score']:.3f}  {m['label']}")

n_matched = sum(1 for m in job_to_skills if m)
print(f"\nJobs with at least 1 skill matched: {n_matched}/{len(jobs_list)}")


[3D animator]
  0.755  3D modelling. three-dimensional modelling, 3D representation, 3D designing, 3D graphics, 3D modeling, 3D rendering. The process of developing a mathematical representation of any three-dimensional surface of an object via specialised software. The product is called a 3D model. It can be displayed as a two-dimensional image through a process called 3D rendering or used in a computer simulation of physical phenomena. The model can also be physically created using 3D printing devices.
  0.755  3D modelling. three-dimensional modelling, 3D representation, 3D designing, 3D graphics, 3D modeling, 3D rendering. The process of developing a mathematical representation of any three-dimensional surface of an object via specialised software. The product is called a 3D model. It can be displayed as a two-dimensional image through a process called 3D rendering or used in a computer simulation of physical phenomena. The model can also be physically created using 3D printing de

### Optional: Tune the threshold

Run this cell to see how many skills get matched at different thresholds — helps you pick the right cutoff.

In [27]:
sim_matrix_js = emb_jobs @ emb_skills.T

print("Threshold  | Avg skills/job | Jobs with 0 skills")
print("-" * 50)
for t in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    per_job = [(sim_matrix_js[i] >= t).sum() for i in range(len(jobs_list))]
    avg = np.mean(per_job)
    zeros = sum(1 for x in per_job if x == 0)
    print(f"  {t:.2f}     |     {avg:6.1f}      |    {zeros}")

Threshold  | Avg skills/job | Jobs with 0 skills
--------------------------------------------------
  0.20     |     2783.4      |    0
  0.25     |     1486.6      |    0
  0.30     |      734.0      |    2
  0.35     |      338.6      |    17
  0.40     |      145.1      |    69
  0.45     |       59.1      |    194
  0.50     |       22.9      |    477


## 5. Match Skills → Courses

For each skill matched to a job, find the courses that cover it.

In [28]:
# Get the unique set of skills we actually need to cover
all_required_skills = list({m["label"] for matches in job_to_skills for m in matches})
print(f"Unique skills to cover: {len(all_required_skills)}")

# Re-embed only the required skills (or reuse from full embedding)
required_skill_indices = [skills_list.index(s) for s in all_required_skills]
emb_required_skills = emb_skills[required_skill_indices]

Unique skills to cover: 3404


In [29]:
skill_to_courses = match_by_threshold(
    emb_required_skills, emb_courses, courses_list,
    threshold=SKILL_COURSE_THRESHOLD,
    max_matches=MAX_COURSES_PER_SKILL
)

# Build a dict skill → courses
skill_course_map = {skill: matches for skill, matches in zip(all_required_skills, skill_to_courses)}

# Quick look
for skill, matches in list(skill_course_map.items())[:3]:
    print(f"\n[{skill}]")
    for m in matches:
        print(f"  {m['score']:.3f}  {m['label']}")


[operate mechanical street sweeping equipment. operating mechanical street sweeping equipment, operate mechanical street sweeper, run mechanical street sweeping equipment, manuever manoeuvre mechanical street sweeping equipment, operate street sweeping equipment. Use and adjust accordingly mechanical equipment such as vacuums, guards, sprayer or water hoses used to eliminate street debris.]
  0.463  Meccanica applicata alle macchine e principi di macchine
  0.463  Meccanica applicata alle macchine e principi di macchine
  0.463  Meccanica applicata alle macchine e principi di macchine
  0.463  Meccanica applicata alle macchine e principi di macchine
  0.463  Meccanica applicata alle macchine e principi di macchine

[produce textile designs. produce designs for textiles, draw sketches for textile designs, produce a textile design, create textile designs. Draw sketches for textile design, by hand or on computer, using specialist Computer Aided Design (CAD) software.]
  0.581  Progettazi

## 6. Build Final Mapping: Job → Courses

Flatten the chain: Job → required skills → covering courses.
Deduplicate courses per job, keep the best score if a course covers multiple skills.

In [30]:
rows = []

for job, skill_matches in zip(jobs_list, job_to_skills):
    # Track best score per course for this job
    course_best_score = {}   # course_name → best score seen
    course_via_skill  = {}   # course_name → skill that linked it

    for sm in skill_matches:
        skill = sm["label"]
        for cm in skill_course_map.get(skill, []):
            course = cm["label"]
            if cm["score"] > course_best_score.get(course, -1):
                course_best_score[course] = cm["score"]
                course_via_skill[course]  = skill

    for course, score in sorted(course_best_score.items(), key=lambda x: -x[1]):
        rows.append({
            "job":         job,
            "course":      course,
            "via_skill":   course_via_skill[course],
            "score":       round(score, 4)
        })

df_result = pd.DataFrame(rows)
print(f"Total job-course pairs: {len(df_result)}")
df_result.head(10)

Total job-course pairs: 32224


,job,course,via_skill,score
0,3D animator,What is graphic design? - Introduction to Grap...,design graphics. apply graphic design techniqu...,0.8183
1,3D animator,Tutorial gratuito di Modellazione 3D - Come Cr...,"3D modelling. three-dimensional modelling, 3D ...",0.7782
2,3D animator,Introducing graphic design - Introduction to G...,design graphics. apply graphic design techniqu...,0.7456
3,3D animator,"Graphic Design Foundations: Ideas, Concepts, a...",design graphics. apply graphic design techniqu...,0.7293
4,3D animator,Aprende AutoCAD 2D y 3D: Básico e Intermedio. ...,operate 3D computer graphics software. use 3D ...,0.6813
5,3D animator,Working with multiple types of media - Graphic...,design graphics. apply graphic design techniqu...,0.6638
6,3D animator,Insegnare Grafica 3D | Udemy,operate 3D computer graphics software. use 3D ...,0.6569
7,3D animator,Stampa 3D | Una guida passo dopo passo | Udemy,apply 3D imaging techniques. create 3D vector ...,0.6368
8,3D animator,Autocad 2d e 3d per il disegno meccanico | Udemy,"rig 3D characters. develop 3D character rigs, ...",0.6269
9,3D animator,3ds max Rigging Character Studio Modulo 1 | Udemy,"rig 3D characters. develop 3D character rigs, ...",0.5897


## 7. Summary Stats

In [31]:
courses_per_job = df_result.groupby("job")["course"].count()
print("Courses per job:")
print(courses_per_job.describe().round(2))
print()

# Jobs with no courses at all
covered_jobs = set(df_result["job"])
uncovered    = [j for j in jobs_list if j not in covered_jobs]
print(f"Jobs with no course matched: {len(uncovered)}")
if uncovered:
    print("  Examples:", uncovered[:5])
    print("  → Try lowering the thresholds for these.")

Courses per job:
count    2969.00
mean       10.85
std         5.57
min         1.00
25%         7.00
50%        10.00
75%        14.00
max        33.00
Name: course, dtype: float64

Jobs with no course matched: 17
  Examples: ['cigar inspector', 'cocoa press operator', 'domestic butler', 'extra', 'farrier']
  → Try lowering the thresholds for these.


## 8. Export

In [33]:
OUTPUT_PATH = "job_course_mapping.xlsx"

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_result.to_excel(writer, sheet_name="Job_Course_Mapping", index=False)

    # Also dump the job→skills intermediate
    skill_rows = []
    for job, matches in zip(jobs_list, job_to_skills):
        for m in matches:
            skill_rows.append({"job": job, "skill": m["label"], "score": round(m["score"], 4)})
    pd.DataFrame(skill_rows).to_excel(writer, sheet_name="Job_Skill_Matching", index=False)

print(f"Saved to {OUTPUT_PATH}")

Saved to job_course_mapping.xlsx
